In [10]:
# ============================================
# 🎯 Timestep-Aware Activation Quantization
# for SDXL Lightning (4-step)
# ============================================

import torch
import torch.nn as nn
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import numpy as np
import time
from pathlib import Path

# ============================================
# 🔢 Quantization Functions
# ============================================

class ActivationQuantizer:
    """Quantize activations to specified bit-width."""
    
    @staticmethod
    def quantize_tensor(tensor, bits=8):
        """
        Quantize a tensor to N-bit representation.
        
        Args:
            tensor: Input tensor (float16/float32)
            bits: Target bit-width (4, 6, 8, 16)
        
        Returns:
            Quantized tensor in original dtype
        """
        if bits == 16:
            # No quantization needed
            return tensor
        
        # Calculate quantization parameters
        qmin = 0
        qmax = 2 ** bits - 1
        
        # Get min/max of tensor
        min_val = tensor.min()
        max_val = tensor.max()
        
        # Calculate scale and zero point
        scale = (max_val - min_val) / qmax
        zero_point = qmin - min_val / scale
        
        # Quantize
        q_tensor = torch.clamp(
            torch.round(tensor / scale + zero_point),
            qmin, qmax
        )
        
        # Dequantize back to original range
        dq_tensor = (q_tensor - zero_point) * scale
        
        return dq_tensor.to(tensor.dtype)


# ============================================
# 📅 Timestep-Aware Quantization Schedule
# ============================================

class TimestepQuantizationSchedule:
    """
    Dynamic quantization schedule based on diffusion timestep.
    
    Strategy for 4-step Lightning model:
    - Step 0 (t=1.0): 4-bit (most aggressive, highest noise)
    - Step 1 (t=0.75): 6-bit (moderate)
    - Step 2 (t=0.5): 8-bit (conservative)
    - Step 3 (t=0.25): 16-bit (full precision, final details)
    """
    
    def __init__(self, num_inference_steps=4, schedule_type="aggressive"):
        self.num_steps = num_inference_steps
        self.schedule_type = schedule_type
        self.current_step = 0
        
        # Define schedules
        self.schedules = {
            "aggressive": [4, 6, 8, 16],      # Max memory savings
            "balanced": [6, 8, 8, 16],        # Balance quality/memory
            "conservative": [8, 8, 16, 16],   # Prioritize quality
            "baseline": [16, 16, 16, 16],     # No quantization
        }
        
        self.schedule = self.schedules.get(schedule_type, self.schedules["balanced"])
        
        # Ensure schedule matches num_steps
        if len(self.schedule) != num_inference_steps:
            # Interpolate schedule to match steps
            self.schedule = self._interpolate_schedule(num_inference_steps)
    
    def _interpolate_schedule(self, target_steps):
        """Interpolate schedule to match target number of steps."""
        original = np.array(self.schedule)
        x_old = np.linspace(0, 1, len(original))
        x_new = np.linspace(0, 1, target_steps)
        interpolated = np.interp(x_new, x_old, original)
        return [int(round(b)) for b in interpolated]
    
    def get_bits_for_step(self, step):
        """Get quantization bits for current step."""
        if step >= len(self.schedule):
            return 16  # Default to full precision
        return self.schedule[step]
    
    def reset(self):
        """Reset step counter."""
        self.current_step = 0
    
    def next_step(self):
        """Advance to next step."""
        self.current_step += 1
        return self.get_bits_for_step(self.current_step)


# ============================================
# 🪝 Forward Hook Implementation
# ============================================

class QuantizationHook:
    """Forward hook to quantize linear layer activations."""
    
    def __init__(self, schedule: TimestepQuantizationSchedule, layer_name: str):
        self.schedule = schedule
        self.layer_name = layer_name
        self.quantizer = ActivationQuantizer()
    
    def __call__(self, module, input, output):
        """
        Hook function called during forward pass.
        
        Args:
            module: The layer being hooked
            input: Input to the layer
            output: Output from the layer
        
        Returns:
            Quantized output
        """
        current_bits = self.schedule.get_bits_for_step(self.schedule.current_step)
        
        # Quantize output
        if isinstance(output, torch.Tensor):
            quantized = self.quantizer.quantize_tensor(output, bits=current_bits)
            return quantized
        elif isinstance(output, tuple):
            # Handle tuple outputs (e.g., attention)
            return tuple(
                self.quantizer.quantize_tensor(o, bits=current_bits) 
                if isinstance(o, torch.Tensor) else o 
                for o in output
            )
        
        return output


# ============================================
# 🔧 Apply Quantization to Model
# ============================================

def apply_quantization_hooks(model, schedule: TimestepQuantizationSchedule, target_layers=None):
    """
    Apply quantization hooks to linear layers in the model.
    
    Args:
        model: The diffusion model (UNet)
        schedule: Quantization schedule
        target_layers: List of layer name patterns to target (None = all linears)
    
    Returns:
        List of hook handles (for cleanup)
    """
    hooks = []
    
    for name, module in model.named_modules():
        # Target linear layers in transformer blocks
        if isinstance(module, nn.Linear):
            # Optional: filter by layer name
            if target_layers is not None:
                if not any(pattern in name for pattern in target_layers):
                    continue
            
            # Create and register hook
            hook = QuantizationHook(schedule, name)
            handle = module.register_forward_hook(hook)
            hooks.append(handle)
            
            print(f"  ✓ Registered hook on: {name}")
    
    print(f"\n📌 Total hooks registered: {len(hooks)}")
    return hooks


def remove_quantization_hooks(hooks):
    """Remove all quantization hooks."""
    for handle in hooks:
        handle.remove()
    print(f"🧹 Removed {len(hooks)} hooks")


# ============================================
# 🎨 Quantization Wrapper with Step Callback
# ============================================

class StepCallback:
    """Callback to track denoising steps and update quantization schedule."""
    
    def __init__(self, schedule):
        self.schedule = schedule
        self.step_count = 0
    
    def __call__(self, pipe, step_index, timestep, callback_kwargs):
        """Called at each denoising step."""
        self.schedule.current_step = step_index
        return callback_kwargs

def enable_quantization(pipe, schedule_type="balanced", num_inference_steps=4):
    """Enable timestep-aware quantization on a pipeline."""
    quantization_schedule = TimestepQuantizationSchedule(
        num_inference_steps=num_inference_steps,
        schedule_type=schedule_type
    )
    
    print(f"\n🔧 Enabling {schedule_type} quantization schedule:")
    print(f"   Steps: {num_inference_steps}")
    print(f"   Bit schedule: {quantization_schedule.schedule}")
    
    # Apply hooks to UNet
    quantization_hooks = apply_quantization_hooks(
        pipe.unet, 
        quantization_schedule,
        target_layers=["attn", "proj"]  # Target attention and projection layers
    )
    
    # Create step callback
    step_callback = StepCallback(quantization_schedule)
    
    # Store references on the pipe object
    pipe._quantization_schedule = quantization_schedule
    pipe._quantization_hooks = quantization_hooks
    pipe._step_callback = step_callback
    
    return pipe

def disable_quantization(pipe):
    """Disable quantization hooks on a pipeline."""
    if hasattr(pipe, '_quantization_hooks') and pipe._quantization_hooks:
        remove_quantization_hooks(pipe._quantization_hooks)
        pipe._quantization_hooks = []
        pipe._quantization_schedule = None
        pipe._step_callback = None


# ============================================
# 📊 Evaluation & Comparison
# ============================================

def compare_quantization_schedules(
    model_path,
    prompt,
    seed=42,
    schedules=["baseline", "conservative", "balanced", "aggressive"],
    num_steps=4,
    device="mps"
):
    """
    Compare different quantization schedules.
    
    Returns:
        Dictionary with results for each schedule
    """
    results = {}
    
    for schedule_type in schedules:
        print("\n" + "="*70)
        print(f"🧪 Testing Schedule: {schedule_type.upper()}")
        print("="*70)
        
        # Load pipeline
        print("Loading model...")
        pipe = StableDiffusionXLPipeline.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            use_safetensors=True,
        )
        pipe = pipe.to(device)
        
        # Enable quantization (skip for baseline)
        if schedule_type != "baseline":
            pipe = enable_quantization(
                pipe,
                schedule_type=schedule_type,
                num_inference_steps=num_steps
            )
        
        # Measure memory before generation
        torch.mps.empty_cache() if device == "mps" else torch.cuda.empty_cache()
        if device == "mps":
            mem_before = torch.mps.current_allocated_memory() / (1024**3)
        elif device == "cuda":
            mem_before = torch.cuda.memory_allocated() / (1024**3)
        
        # Generate image
        print(f"\n🎨 Generating image...")
        generator = torch.Generator(device="cpu").manual_seed(seed)
        
        # Reset schedule before generation
        if hasattr(pipe, '_quantization_schedule') and pipe._quantization_schedule:
            pipe._quantization_schedule.reset()
        
        # Get callback if it exists
        callback = None
        if hasattr(pipe, '_step_callback'):
            callback = pipe._step_callback
        
        start_time = time.time()
        image = pipe(
            prompt=prompt,
            num_inference_steps=num_steps,
            guidance_scale=0.0,
            generator=generator,
            height=512,
            width=512,
            callback_on_step_end=callback,  # Add callback to track steps
        ).images[0]
        inference_time = time.time() - start_time
        
        # Measure peak memory
        if device == "mps":
            mem_after = torch.mps.current_allocated_memory() / (1024**3)
        elif device == "cuda":
            mem_after = torch.cuda.max_memory_allocated() / (1024**3)
        
        peak_memory = mem_after
        
        # Get schedule info
        if hasattr(pipe, '_quantization_schedule') and pipe._quantization_schedule:
            schedule_info = pipe._quantization_schedule.schedule
        else:
            schedule_info = [16] * num_steps
        
        results[schedule_type] = {
            "image": image,
            "inference_time": inference_time,
            "peak_memory_gb": peak_memory,
            "schedule": schedule_info,
        }
        
        print(f"✅ Done in {inference_time:.2f}s")
        print(f"📊 Peak memory: {peak_memory:.2f} GB")
        
        # Cleanup
        if schedule_type != "baseline":
            disable_quantization(pipe)
        del pipe
        torch.mps.empty_cache() if device == "mps" else torch.cuda.empty_cache()
    
    return results


# ============================================
# 🚀 Example Usage
# ============================================

if __name__ == "__main__":
    MODEL_PATH = "quantized_models/sdxl_lightning_4step_mps_fp16"
    PROMPT = "A serene mountain landscape at sunset with golden light"
    
    # Run comparison
    results = compare_quantization_schedules(
        model_path=MODEL_PATH,
        prompt=PROMPT,
        seed=42,
        schedules=["baseline", "conservative", "balanced", "aggressive"],
        num_steps=4,
        device="mps"
    )
    
    # Print summary
    print("\n" + "="*70)
    print("📋 QUANTIZATION SCHEDULE COMPARISON")
    print("="*70)
    
    baseline_time = results["baseline"]["inference_time"]
    baseline_mem = results["baseline"]["peak_memory_gb"]
    
    for schedule_name, data in results.items():
        time_speedup = baseline_time / data["inference_time"]
        mem_savings = baseline_mem - data["peak_memory_gb"]
        mem_savings_pct = (mem_savings / baseline_mem) * 100
        
        print(f"\n{schedule_name.upper()}:")
        print(f"  Bit schedule: {data['schedule']}")
        print(f"  Inference time: {data['inference_time']:.2f}s ({time_speedup:.2f}x)")
        print(f"  Peak memory: {data['peak_memory_gb']:.2f} GB (saves {mem_savings:.2f} GB / {mem_savings_pct:.1f}%)")
        
        # Save image
        output_path = Path(f"./evaluation_outputs/quantized_{schedule_name}.png")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        data["image"].save(output_path)
        print(f"  💾 Saved: {output_path}")


🧪 Testing Schedule: BASELINE
Loading model...


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  9.87it/s]



🎨 Generating image...


100%|██████████| 4/4 [00:03<00:00,  1.14it/s]


✅ Done in 8.78s
📊 Peak memory: 6.47 GB

🧪 Testing Schedule: CONSERVATIVE
Loading model...


...

✅ Done in 9.96s
📊 Peak memory: 6.47 GB
🧹 Removed 669 hooks

🧪 Testing Schedule: AGGRESSIVE
Loading model...


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 10.38it/s]



🔧 Enabling aggressive quantization schedule:
   Steps: 4
   Bit schedule: [4, 6, 8, 16]
  ✓ Registered hook on: down_blocks.0.resnets.0.time_emb_proj
  ✓ Registered hook on: down_blocks.0.resnets.1.time_emb_proj
  ✓ Registered hook on: down_blocks.1.attentions.0.proj_in
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_q
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_k
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_v
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn1.to_out.0
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_q
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_k
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_v
  ✓ Registered hook on: down_blocks.1.attentions.0.transformer_blocks.0.attn2.to_out.0
  ✓ Registered hook on: down_blocks.1.attentions.0.trans

100%|██████████| 4/4 [00:04<00:00,  1.08s/it]


✅ Done in 6.81s
📊 Peak memory: 6.47 GB
🧹 Removed 669 hooks

📋 QUANTIZATION SCHEDULE COMPARISON

BASELINE:
  Bit schedule: [16, 16, 16, 16]
  Inference time: 8.78s (1.00x)
  Peak memory: 6.47 GB (saves 0.00 GB / 0.0%)
  💾 Saved: evaluation_outputs/quantized_baseline.png

CONSERVATIVE:
  Bit schedule: [8, 8, 16, 16]
  Inference time: 9.20s (0.95x)
  Peak memory: 6.47 GB (saves -0.00 GB / -0.1%)
  💾 Saved: evaluation_outputs/quantized_conservative.png

BALANCED:
  Bit schedule: [6, 8, 8, 16]
  Inference time: 9.96s (0.88x)
  Peak memory: 6.47 GB (saves -0.00 GB / -0.1%)
  💾 Saved: evaluation_outputs/quantized_balanced.png

AGGRESSIVE:
  Bit schedule: [4, 6, 8, 16]
  Inference time: 6.81s (1.29x)
  Peak memory: 6.47 GB (saves -0.00 GB / -0.1%)
  💾 Saved: evaluation_outputs/quantized_aggressive.png


In [9]:
import os
from pathlib import Path

# Check current directory
print(f"Current directory: {os.getcwd()}")

# List what's in the current directory
print(f"\nContents of current directory:")
for item in os.listdir('.'):
    print(f"  - {item}")

# Check if sdxl folder exists
if os.path.exists('sdxl'):
    print(f"\n✓ sdxl/ folder found")
    print(f"Contents of sdxl/:")
    for item in os.listdir('sdxl'):
        print(f"  - {item}")
    
    if os.path.exists('sdxl/quantized_models'):
        print(f"\n✓ sdxl/quantized_models/ folder found")
        print(f"Contents of sdxl/quantized_models/:")
        for item in os.listdir('sdxl/quantized_models'):
            print(f"  - {item}")

# Try to find the model
possible_paths = [
    "sdxl/quantized_models/sdxl_lightning_4step_mps_fp16",
    "./sdxl/quantized_models/sdxl_lightning_4step_mps_fp16",
    "../sdxl/quantized_models/sdxl_lightning_4step_mps_fp16",
]

print(f"\n🔍 Checking possible paths:")
for p in possible_paths:
    exists = os.path.exists(p)
    print(f"  {p}: {'✓ EXISTS' if exists else '✗ NOT FOUND'}")
    if exists:
        print(f"    → Use this path: MODEL_PATH = '{p}'")

Current directory: /Users/yash.honrao/Documents/Fall 25/GenAI/Project/sdxl

Contents of current directory:
  - model_evaluation.ipynb
  - .DS_Store
  - compare_fp32_fp16.py
  - requirements.txt
  - generated_images
  - eval "$(ssh-agent -s)"
  - evaluation_outputs
  - memory_requirements.ipynb
  - time-step aware quantization.ipynb
  - comparison_fp16_512x512.png
  - .gitignore
  - model_quantization.py
  - output.png
  - heavy_model.py
  - original_huggingface_model.py
  - .gitattributes
  - model_evaluation_old.ipynb
  - inference.py
  - quantized_models
  - inference.ipynb
  - comparison_fp32_512x512.png
  - eval "$(ssh-agent -s)".pub

🔍 Checking possible paths:
  sdxl/quantized_models/sdxl_lightning_4step_mps_fp16: ✗ NOT FOUND
  ./sdxl/quantized_models/sdxl_lightning_4step_mps_fp16: ✗ NOT FOUND
  ../sdxl/quantized_models/sdxl_lightning_4step_mps_fp16: ✓ EXISTS
    → Use this path: MODEL_PATH = '../sdxl/quantized_models/sdxl_lightning_4step_mps_fp16'
